In [ ]:
!pip install lapjv
!pip install igraph
!pip install leidenalg

import numpy as np
import scipy
efrom lapjv import lapjv
from scipy import stats
import time
from itertools import combinations
import scipy.io
import pandas as pd
import csv
from google.colab import drive
from collections import defaultdict
from os import cpu_count
import re
from collections import defaultdict
import matplotlib.pyplot as plt
import igraph as ig
import leidenalg as la
import random
from typing import List



In [ ]:
drive.mount('/content/drive')
#%cd /content/drive/My\ Drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/My\ Drive

/content/drive/My Drive


In [ ]:
# All helper methods used for data processing

def has_duplicate_2d(array):
    seen = set()
    for row in array:
        for elem in row:
            if elem in seen:
                return True
            seen.add(elem)
    return False

def count_singleton_lists(d):
    total_values = len(d)
    count_singleton = sum(1 for v in d.values() if isinstance(v, list) and len(v) == 1)
    ratio = count_singleton / total_values if total_values > 0 else 0
    return count_singleton, ratio

def generate_unique_pairs(arr):
    return list(combinations(arr, 2))

def merge_single_element_sublists(lst):
    singles = []
    result = []
    for sub in lst:
        if len(sub) == 1:
            singles.extend(sub)
        else:
            result.append(sub)
    if singles:
        result.append(singles)
    return result

def barycenter_matrices(n, m):
    k = len(n)
    cum_n = np.cumsum(n)
    cum_n = [0] + cum_n.tolist()

    PP = np.zeros((cum_n[-1], cum_n[-1]));
    P = np.zeros((cum_n[-1], cum_n[-1]));
    for i in range(0, k):
        n_u_i = n[i]-m[i]
        P[cum_n[i]+m[i]:cum_n[i+1], cum_n[i]+m[i]:cum_n[i+1]] = np.ones((n_u_i, n_u_i))/n_u_i;
        if m[i] == 0:
            PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]] =  P[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]]
        else:
            PP[cum_n[i]:cum_n[i]+m[i], cum_n[i]:cum_n[i]+m[i]] = np.identity(m[i]);
            PP[(cum_n[i]+m[i]):cum_n[i+1], (cum_n[i]+m[i]):cum_n[i+1]] = np.ones((n_u_i, n_u_i))/n_u_i;
    return P, PP

def group_indices(arr):
  unique_vals = np.unique(arr)
  index_groups = {}

  for val in unique_vals:
    indices = np.where(arr == val)[0].tolist()
    index_groups[val] = indices
  return index_groups

def generate_array_comm(n, k):
    base_value = n // k
    remainder = n % k
    result = [base_value] * k

    for i in range(remainder):
        result[i] += 1

    return result

def new_generate_array_comm(l, x):
    total = sum(l)
    ratio = x / total
    l2 = [min(int(size * ratio), size) for size in l]
    for i in range(len(l)):
        if l2[i] == l[i]:
            if l2[i] > 0:
                l2[i] -= 1
            else:
                l2[i] += 1
    current_sum = sum(l2)
    difference = x - current_sum

    fractional_parts = [(size * ratio) - int(size * ratio) for size in l]
    indices_sorted = sorted(
        range(len(l)),
        key=lambda i: (fractional_parts[i], -l[i]),
        reverse=True
    )

    for i in indices_sorted:
        if difference <= 0:
            break
        if abs(l2[i] + 1 - l[i]) >= 1 and l2[i] < l[i]:
            l2[i] += 1
            difference -= 1

    return l2


def direct_sum(A, B):
    return np.block([
        [A, np.zeros((A.shape[0], B.shape[1]))],
        [np.zeros((B.shape[0], A.shape[1])), B]
    ])

def label_to_numbers(labels):
    label_mapping = {}
    numbered_labels = []

    for label in labels:
        if label not in label_mapping:
            label_mapping[label] = len(label_mapping)
        numbered_labels.append(label_mapping[label])

    return numbered_labels, label_mapping

def relabel_alphebatically(mapping, label_list):
    sorted_keys = sorted(list(mapping.keys()))
    alphebatical_labels = [mapping[key] for key in sorted_keys if key in mapping]
    perm = np.argsort(alphebatical_labels)
    new_mapping = dict(zip(sorted_keys, list(range(0,len(sorted_keys)))))
    new_label_list = [int(perm[i]) for i in label_list]
    return new_label_list, new_mapping

def right_perm(mapping, length):
    mapped_comm = map_keys(mapping, list(range(0,length)))
    ind = np.argsort(mapped_comm)
    ind2 = np.argsort(ind)
    ind = [int(x) for x in ind]
    ind2 = [int(x) for x in ind2]
    return mapped_comm, ind, ind2

def merge_ones(lst):
    i = 0
    while i < len(lst):
        if lst[i] == 1:
            if i == 0:
                lst[i + 1] += lst[i]
            elif i == len(lst) - 1:
                lst[i - 1] += lst[i]
            else:
                lst[i + 1] += lst[i]
            lst.pop(i)
            i = max(0, i - 1)
        else:
            i += 1
    return lst

def acc(corr, perm, num_seeds, total):
    return ((sum(1 for a, b in zip(perm, corr) if a == b))-num_seeds)/(total-num_seeds)

def permute_communities(A, B, D):
    groups = []
    current = 0
    for size in B:
        groups.append(A[current:current + size])
        current += size

    result = []
    for key in sorted(D.keys()):
        for b_index in D[key]:
            result.extend(groups[b_index])

    return result

def get_var_name(variable):
     for name, value in globals().items():
        if value is variable:
            return name

def label_to_numbers(labels):
    label_mapping = {}
    numbered_labels = []

    for label in labels:
        if label not in label_mapping:
            label_mapping[label] = len(label_mapping)
        numbered_labels.append(label_mapping[label])

    return numbered_labels, label_mapping

def find_all_occurrences(input_list):
    unique_indices = defaultdict(list)
    for index, value in enumerate(input_list):
        unique_indices[value].append(index)
    return dict(unique_indices)

def map_correspondence(list1, list2):
    if len(list1) != len(list2):
        raise ValueError("Both lists must have the same length")

    correspondence = {}
    visited = set()
    for key, value in zip(list1, list2):
        if key not in correspondence:
            correspondence[key] = set()
        if value not in visited:
            visited.add(value)
            correspondence[key].add(value)

    return {k: sorted(v) for k, v in correspondence.items()}

def group_array_by_ind_element(arr, ind):
  grouped_data = {}
  for entry in arr:
    element = entry[ind]
    if element not in grouped_data:
      grouped_data[element] = []
    grouped_data[element].append(entry)
  return grouped_data

def outer_product(X):
    return np.dot(X.T, X)

def read_file(to_read):
    matrix = np.empty((0,992))
    first = True
    pData = pd.read_csv("pData.csv", index_col=-1)
    with open(to_read, 'r') as file:
        reader = csv.reader(file)
        for row in reader:
            if first:
                row_num = np.array([float(num) for num in row[0].split()])
                indices = np.vstack([list(pData['index ']), row_num])
                first = False
            else:
                row_num = np.array([float(num) for num in row[0].split()])
                matrix = np.vstack([matrix, row_num])
    return matrix

def map_keys(d, Arr):
    element_to_key = {}
    for key, values in d.items():
        for val in values:
            element_to_key[val] = key
    return [element_to_key[num] for num in Arr]

def permute_array(arr, perm):
    result = [0] * len(arr)

    for i in range(len(arr)):
        result[i] = arr[perm[i]]

    return result

def get_comm_size(arr):
    sizes = []
    count = 1

    for i in range(1, len(arr)):
        if arr[i] == arr[i - 1]:
            count += 1
        else:
            sizes.append(count)
            count = 1

    sizes.append(count)
    return sizes

def generate_seed_in_comm(comm_size, seed_num):
    seed_in_comm = [0] * len(comm_size)

    if seed_num < len(comm_size):
        seed_in_comm[:seed_num] = [1] * seed_num
    else:
        seed_in_comm = [1] * len(comm_size)
        current_sum = len(comm_size)
        max_possible = sum(size - 1 for size in comm_size)

        if seed_num == 0:
            seed_in_comm = [0] * len(comm_size)

        remaining = seed_num - current_sum
        indices = [i for i in range(len(comm_size)) if comm_size[i] > 1]

        while remaining > 0 and indices:
            per_element = max(1, remaining // len(indices))

            for i in indices.copy():
                alloc = min(per_element, comm_size[i] - 1 - seed_in_comm[i], remaining)
                seed_in_comm[i] += alloc
                remaining -= alloc

                if seed_in_comm[i] >= comm_size[i] - 1:
                    indices.remove(i)


    final_perm = []
    prev = 0
    for index, value in enumerate(seed_in_comm):
        final_perm = final_perm + list(range(prev,prev+seed_in_comm[index]))
        final_perm = final_perm + [int(x) for x in list(np.random.permutation(range(prev+seed_in_comm[index], prev+comm_size[index])))]
        prev = prev + comm_size[index]
    final_perm = [int(x) for x in final_perm]

    non_comm_perm = list(range(0,sum(seed_in_comm))) + [int(x) for x in list(np.random.permutation(range(sum(seed_in_comm), sum(comm_size))))]

    return seed_in_comm, final_perm, non_comm_perm


def shuffle_percentage(arr, percentage):
    n = len(arr)
    count = int(n * percentage / 100)
    if count == 0:
        return arr
    indices_to_shuffle = random.sample(range(n), count)
    elements_to_shuffle = [arr[i] for i in indices_to_shuffle]
    random.shuffle(elements_to_shuffle)
    for i, idx in enumerate(indices_to_shuffle):
        arr[idx] = elements_to_shuffle[i]
    return arr

def leiden_communities_to_labels(communities):
    n = max(max(comm) for comm in communities) + 1
    labels = [None] * n
    for community_idx, nodes in enumerate(communities):
        for node in nodes:
            labels[node] = community_idx
    return labels



In [ ]:
# block_SGM function
# n -- a vec of partition sizes
# m -- a vec of seeds per partition
# It is assumed that the first m(i) vertices of partiton i
# correspond respectively to the first m(i) vertices of partition i of B,


def block_SGM(A, B, n, m, max_iter, tol):

    P, PP = barycenter_matrices(n, m);
    k = len(n);
    patience = max_iter;
    iter = 0;
    go_on = 1;
    cum_n = np.cumsum(n);
    cum_n = [0] + cum_n.tolist();
    u = [n[i] - m[i] for i in range(k)];


    while iter < max_iter and go_on == 1:

        old_P = PP;
        order = range(0,k)

        for ind in range(k):
            j = order[ind]
            grad_j = np.zeros((n[j], n[j]))
            for i in range(k):
                if i == j:
                    grad_j = grad_j + A[cum_n[i]:cum_n[i+1],cum_n[i]:cum_n[i+1]] \
                    @ PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]] \
                    @ (B[cum_n[i]:cum_n[i+1],cum_n[i]:cum_n[i+1]]).T \
                    + (A[cum_n[i]:cum_n[i+1],cum_n[i]:cum_n[i+1]]).T \
                    @ PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]] @ (B[cum_n[i]:cum_n[i+1],cum_n[i]:cum_n[i+1]])

                else:
                    grad_j = grad_j + 2*((A[cum_n[i]:cum_n[i+1],cum_n[j]:cum_n[j+1]]).T\
                    @ PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]]\
                    @ B[cum_n[i]:cum_n[i+1], cum_n[j]:cum_n[j+1]])

            #make round a flag
            cost_matrix = -np.round(grad_j[m[j]:, m[j]:] , 5)
            col_ind, _, _ = lapjv(cost_matrix)
            Tj = np.eye(n[j]-m[j])[col_ind, :]
            Tj = direct_sum(np.eye(m[j]), Tj)


            R = PP.copy()
            R[cum_n[j]:cum_n[j+1], cum_n[j]:cum_n[j+1]] = Tj

            T1 = (A @ PP @ B.T @ PP.T).diagonal().sum()
            T2 = (A @ PP @ B.T @ R.T).diagonal().sum()
            T3 = (A @ R @ B.T @ PP.T).diagonal().sum()
            T4 = (A @ R @ B.T @ R.T).diagonal().sum()

            if (2 * (T1 - T2 - T3 + T4)) == 0:
                PP = PP;
            else:
                beta = (-T2 - T3 + 2 * T4) / (2 * (T1 - T2 - T3 + T4))

                Qj = Tj.copy()

                P_new_potential = beta * PP[cum_n[j]:cum_n[j+1], cum_n[j]:cum_n[j+1]] + (1-beta)*Qj

                PP_new_potential = PP.copy()
                PP_new_potential[cum_n[j]:cum_n[j+1], cum_n[j]:cum_n[j+1]] = P_new_potential

                PP_new_potential_beta0 = PP.copy()
                PP_new_potential_beta0[cum_n[j]:cum_n[j+1], cum_n[j]:cum_n[j+1]] = Qj


                obj_beta0 =  -(A @ PP_new_potential_beta0 @ B.T @ PP_new_potential_beta0.T).diagonal().sum()

                obj_potential = -(A @ PP_new_potential @ B.T @ PP_new_potential.T).diagonal().sum()
                obj_now = -(A @ PP @ B.T @ PP.T).diagonal().sum()

                #print(beta)
                if beta > 1 or beta < 0:
                    if obj_beta0 < obj_now:
                        PP = PP_new_potential_beta0
                        #print("step 0")
                    #else:
                    #    print("step 1")

                if beta <=1 and beta >= 0:
                    if obj_potential < obj_now and obj_potential < obj_beta0:
                        PP = PP_new_potential
                        #print("step beta")
                    elif obj_potential > obj_beta0 and obj_now > obj_beta0:
                        PP = PP_new_potential_beta0
                        #print("step 0")
                    #else:
                    #    print("step 1")

        iter += 1

        s = np.sum(np.abs(old_P - PP))

        if s < 1 - tol:
            #print("it ends with s < 1 - tol")
            go_on = 0

    corr = [None] * cum_n[-1]

    # make corr vector for each group
    for i in range(k):

            P[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]] = np.round(PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]],5)
            col_ind, _, _ = lapjv(-P[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]])

            for j in range(n[i]):
                if j < m[i]:
                    corr[cum_n[i] + j] = cum_n[i] + j
                else:
                    corr[cum_n[i] + j] = cum_n[i] + col_ind[j]

    #print("This one uses "+str(iter)+"iterations")

    return corr

In [ ]:
# block_SGM function
# n -- a vec of partition sizes
# m -- a vec of seeds per partition
# It is assumed that the first m(i) vertices of partiton i
# correspond respectively to the first m(i) vertices of partition i of B,


def block_SGM_iter(A, B, n, m, max_iter, tol):
    corrs = {}

    P, PP = barycenter_matrices(n, m);
    k = len(n);
    patience = max_iter;
    iter = 0;
    go_on = 1;
    cum_n = np.cumsum(n);
    cum_n = [0] + cum_n.tolist();
    u = [n[i] - m[i] for i in range(k)];


    while iter < max_iter and go_on == 1:

        old_P = PP;
        order = range(0,k)

        for ind in range(k):
            j = order[ind]
            grad_j = np.zeros((n[j], n[j]))
            for i in range(k):
                if i == j:
                    grad_j = grad_j + A[cum_n[i]:cum_n[i+1],cum_n[i]:cum_n[i+1]] \
                    @ PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]] \
                    @ (B[cum_n[i]:cum_n[i+1],cum_n[i]:cum_n[i+1]]).T \
                    + (A[cum_n[i]:cum_n[i+1],cum_n[i]:cum_n[i+1]]).T \
                    @ PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]] @ (B[cum_n[i]:cum_n[i+1],cum_n[i]:cum_n[i+1]])

                else:
                    grad_j = grad_j + 2*(A[cum_n[i]:cum_n[i+1],cum_n[j]:cum_n[j+1]]).T\
                    @ PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]]\
                    @ B[cum_n[i]:cum_n[i+1], cum_n[j]:cum_n[j+1]]

            #make round a flag
            cost_matrix = -np.round(grad_j[m[j]:, m[j]:] , 5)
            col_ind, _, _ = lapjv(cost_matrix)
            Tj = np.eye(n[j]-m[j])[col_ind, :]
            Tj = direct_sum(np.eye(m[j]), Tj)


            R = PP.copy()
            R[cum_n[j]:cum_n[j+1], cum_n[j]:cum_n[j+1]] = Tj

            T1 = (A @ PP @ B.T @ PP.T).diagonal().sum()
            T2 = (A @ PP @ B.T @ R.T).diagonal().sum()
            T3 = (A @ R @ B.T @ PP.T).diagonal().sum()
            T4 = (A @ R @ B.T @ R.T).diagonal().sum()

            if (2 * (T1 - T2 - T3 + T4)) == 0:
                PP = PP;
            else:
                beta = (-T2 - T3 + 2 * T4) / (2 * (T1 - T2 - T3 + T4))

                Qj = Tj.copy()

                P_new_potential = beta * PP[cum_n[j]:cum_n[j+1], cum_n[j]:cum_n[j+1]] + (1-beta)*Qj

                PP_new_potential = PP.copy()
                PP_new_potential[cum_n[j]:cum_n[j+1], cum_n[j]:cum_n[j+1]] = P_new_potential

                PP_new_potential_beta0 = PP.copy()
                PP_new_potential_beta0[cum_n[j]:cum_n[j+1], cum_n[j]:cum_n[j+1]] = Qj


                obj_beta0 =  -(A @ PP_new_potential_beta0 @ B.T @ PP_new_potential_beta0.T).diagonal().sum()

                obj_potential = -(A @ PP_new_potential @ B.T @ PP_new_potential.T).diagonal().sum()
                obj_now = -(A @ PP @ B.T @ PP.T).diagonal().sum()

                #print(beta)
                if beta > 1 or beta < 0:
                    if obj_beta0 < obj_now:
                        PP = PP_new_potential_beta0
                        #print("step 0")
                    #else:
                    #    print("step 1")

                if beta <=1 and beta >= 0:
                    if obj_potential < obj_now and obj_potential < obj_beta0:
                        PP = PP_new_potential
                        #print("step beta")
                    elif obj_potential > obj_beta0 and obj_now > obj_beta0:
                        PP = PP_new_potential_beta0
                        #print("step 0")
                    #else:
                    #    print("step 1")

        iter += 1

        s = np.sum(np.abs(old_P - PP))

        if s < 1 - tol:
            #print("it ends with s < 1 - tol")
            go_on = 0

        this_corr = [None] * cum_n[-1]
        for i in range(k):

            P[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]] = np.round(PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]],5)
            col_ind, _, _ = lapjv(-P[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]])

            for j in range(n[i]):
                if j < m[i]:
                    this_corr[cum_n[i] + j] = cum_n[i] + j
                else:
                    this_corr[cum_n[i] + j] = cum_n[i] + col_ind[j]
        corrs[iter] = this_corr

    corr = [None] * cum_n[-1]

    # make corr vector for each group
    for i in range(k):

            P[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]] = np.round(PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]],5)
            col_ind, _, _ = lapjv(-P[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]])

            for j in range(n[i]):
                if j < m[i]:
                    corr[cum_n[i] + j] = cum_n[i] + j
                else:
                    corr[cum_n[i] + j] = cum_n[i] + col_ind[j]

    print("This one uses "+str(iter)+"iterations")

    return corr, corrs

In [ ]:
import numpy as np

def shuffle_with_normal_prob(arr, mean=None, std=1):
    arr = np.array(arr)
    n = len(arr)
    indices = np.arange(n)

    # Set center of the normal distribution
    if mean is None:
        mean = (n - 1) / 2  # Default to center

    # Compute the normal probability for each index
    probs = np.exp(-0.5 * ((indices - mean) / std)**2)
    probs /= probs.max()  # Normalize so the max probability is 1 (can adjust).

    # Decide for each node if it's to be shuffled
    shuffle_mask = np.random.rand(n) < probs

    shuffled_arr = arr.copy()
    # Only shuffle those selected by the mask
    selected_indices = np.where(shuffle_mask)[0]
    selected_elements = shuffled_arr[selected_indices]
    np.random.shuffle(selected_elements)
    shuffled_arr[selected_indices] = selected_elements
    print("shuffle percentage")
    print(sum(el1 != el2 for el1, el2 in zip(list(range(0,1382)), shuffled_arr.tolist()))/1382)
    return shuffled_arr.tolist()

In [ ]:
!pip install python-louvain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 9.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for python-louvain: filename=python_louvain-0.16-py3-none-any.whl size=9460 sha256=cabcb1af825eb5631d247f6d4fa72c7fe78f841582dd35c9c714e34e44ad3a70
  Stored in directory: /root/.cache/pip/wheels/40/f1/e3/485b698c520fa0baee1d07897abc7b8d6479b7d199ce96f4af
Successfully built python-louvain


In [ ]:
import community as community_louvain  # This is the python-louvain package
G = nx.from_numpy_array(graphB)
partition = community_louvain.best_partition(G)
print(partition)

{0: 0, 1: 0, 2: 1, 3: 2, 4: 0, 5: 3, 6: 0, 7: 1, 8: 6, 9: 3, 10: 0, 11: 0, 12: 1, 13: 0, 14: 0, 15: 0, 16: 3, 17: 1, 18: 1, 19: 3, 20: 5, 21: 0, 22: 0, 23: 0, 24: 3, 25: 2, 26: 6, 27: 0, 28: 0, 29: 3, 30: 3, 31: 5, 32: 5, 33: 0, 34: 0, 35: 2, 36: 0, 37: 1, 38: 5, 39: 1, 40: 3, 41: 1, 42: 0, 43: 3, 44: 6, 45: 1, 46: 0, 47: 5, 48: 1, 49: 0, 50: 3, 51: 0, 52: 1, 53: 0, 54: 0, 55: 1, 56: 0, 57: 1, 58: 1, 59: 0, 60: 0, 61: 0, 62: 0, 63: 0, 64: 5, 65: 0, 66: 0, 67: 0, 68: 0, 69: 0, 70: 2, 71: 0, 72: 5, 73: 5, 74: 0, 75: 0, 76: 0, 77: 0, 78: 0, 79: 0, 80: 0, 81: 0, 82: 0, 83: 0, 84: 0, 85: 0, 86: 0, 87: 0, 88: 0, 89: 0, 90: 0, 91: 0, 92: 0, 93: 0, 94: 0, 95: 0, 96: 0, 97: 0, 98: 0, 99: 0, 100: 0, 101: 0, 102: 0, 103: 0, 104: 0, 105: 0, 106: 0, 107: 0, 108: 0, 109: 0, 110: 0, 111: 0, 112: 0, 113: 0, 114: 0, 115: 0, 116: 0, 117: 0, 118: 0, 119: 0, 120: 0, 121: 0, 122: 0, 123: 0, 124: 0, 125: 0, 126: 0, 127: 0, 128: 0, 129: 0, 130: 0, 131: 0, 132: 0, 133: 0, 134: 0, 135: 0, 136: 0, 137: 0, 138: 

In [ ]:
louvain = [partition[x] for x in range(1382)]
louvain_indices = np.array(louvain).argsort()

newman = find_subarray_indices(first_level_communities)
newman_indices = np.array(newman).argsort()

In [ ]:
from collections import Counter
louvain_comm = [(Counter(louvain))[x] for x in range(max(louvain)+1)]
newman_comm = [(Counter(newman))[x] for x in range(max(newman)+1)]

In [ ]:
louvain_comm = [285,
 417,
 203,
 113,
 134, 128, 32, 11,59]

In [ ]:
# vanilla elegans experiments
elegans = scipy.io.loadmat('elegansGraph.mat')
graphA = elegans['Achem'].toarray()
graphB = elegans['Agap'].toarray()
elegans_labels = scipy.io.loadmat('celegans_labels.mat')

indices = elegans_labels['cel_labels'].argsort()

graphA = graphA = elegans['Achem'].toarray()[indices][:, indices]
graphB = graphB = elegans['Agap'].toarray() [indices][:, indices]

graphA2 = graphA.copy()

avg_arr = []

comm_len = [np.sum(elegans_labels['cel_labels'] == i) for i in range(1, 4)]
acc_dict = {}

for percentage in range(0,1,1):
      acc_arr_per = []
      shuffled_perm = shuffle_percentage(list(range(0,279)), percentage)
      print("-------------------------------------------------------")
      print("Current percentage of shuffling:" + str(percentage))
      avg = 0
      for i in [0,1,5,10,20,50,75,100,150,200]:

            print("-------------------------------------------------------")
            print("Current number of seeds:" + str(i))
            avg = 0
            seeds_in_comm2 = generate_array_comm(i, 3)
            for j in range(0,100):
                  cum_comm = np.cumsum(comm_len)
                  cum_comm = [0] + cum_comm.tolist()
                  perm = np.array(range(0,cum_comm[-1]))


                  for i in range(0, len(seeds_in_comm2)):
                      slice_size = cum_comm[i+1] - (cum_comm[i] + seeds_in_comm2[i])
                      permuted_indices = np.random.permutation(slice_size)
                      shifted_indices = permuted_indices + (cum_comm[i] + seeds_in_comm2[i])
                      perm[(cum_comm[i]+seeds_in_comm2[i]):cum_comm[i+1]] = shifted_indices

                  graphA3 = graphA.copy()
                  graphA3[:] = graphA3[shuffled_perm][:, shuffled_perm]
                  graphA2[:] = graphA3[perm][:, perm]
                  #print(perm)
                  corr, corrs = block_SGM_iter(graphA2, graphB, comm_len, seeds_in_comm2, 20 , 1-10**(-6));

                  num_seeds = np.sum(seeds_in_comm2)
                  avg = avg+ ((np.sum(perm == corr)-num_seeds)/(279-num_seeds))
                  print((np.sum(perm == corr)-num_seeds)/(279-num_seeds))

            avg_arr = avg_arr + [avg/10]
            acc_arr_per = acc_arr_per + [avg/10]
            print("avg result for this number of seeds is"+str(avg/10))
            print("-------------------------------------------------------")
            if percentage not in acc_dict.keys():
                  acc_dict[percentage] = acc_arr_per
            else:
                  acc_dict[percentage].append(avg/10)
            print("Current acc dict")
            print(acc_dict)


-------------------------------------------------------
Current percentage of shuffling:0
-------------------------------------------------------
Current number of seeds:0
This one uses 20iterations
0.025089605734767026
This one uses 20iterations
0.010752688172043012
This one uses 18iterations
0.014336917562724014
This one uses 14iterations
0.025089605734767026
This one uses 20iterations
0.021505376344086023
This one uses 20iterations
0.021505376344086023
This one uses 20iterations
0.021505376344086023
This one uses 18iterations
0.017921146953405017
This one uses 20iterations
0.014336917562724014
This one uses 13iterations
0.007168458781362007
This one uses 20iterations
0.010752688172043012
This one uses 11iterations
0.010752688172043012
This one uses 20iterations
0.014336917562724014
This one uses 20iterations
0.007168458781362007
This one uses 18iterations
0.017921146953405017
This one uses 20iterations
0.010752688172043012
This one uses 17iterations
0.025089605734767026
This one use

In [ ]:
# regular leiden

import numpy as np
import leidenalg as la
import igraph as ig
from sklearn.cluster import AgglomerativeClustering
from collections import defaultdict
from itertools import groupby

elegans = scipy.io.loadmat('elegansGraph.mat')
graphA = elegans['Achem'].toarray()
graphB = elegans['Agap'].toarray()
graphA2 = graphA.copy()

graph = ig.Graph.Adjacency(graphB.tolist(), mode="undirected")
N = graph.vcount()
n_runs = 100

all_leiden_elegans = []

for i in range(n_runs):
    part = list(la.find_partition(graph, la.ModularityVertexPartition, seed=i))
    all_leiden_elegans.append(part)

#all_leiden_elegans = [np.asarray(part) for part in all_leiden_elegans]
np.save('100runs_leiden_elegans_with_singletons.npy', np.array(all_leiden_elegans, dtype=object), allow_pickle=True)


In [ ]:
all_leiden_elegans_here = np.load('100runs_leiden_elegans_with_singletons.npy', allow_pickle=True)

In [ ]:
all_leiden_elegans_here = np.load('100runs_leiden_elegans_with_singletons.npy', allow_pickle=True)
all_leiden_elegans_labels = []
for i in range(100):
    all_leiden_elegans_labels.append(leiden_communities_to_labels(all_leiden_elegans_here[i]))

np.save('100runs_leiden_elegans_with_singletons_labels.npy', np.array(all_leiden_elegans_labels, dtype=object), allow_pickle=True)

In [ ]:
# leiden with singletons elegans experiments
elegans = scipy.io.loadmat('elegansGraph.mat')
graphA = elegans['Achem'].toarray()
graphB = elegans['Agap'].toarray()


elegans_labels_leiden = np.load('100runs_leiden_elegans_with_singletons_labels.npy', allow_pickle=True)

acc_dict = {}

for runs in range(10,100,1):

    indices = elegans_labels_leiden[runs].argsort()

    graphA = elegans['Achem'].toarray()[indices][:, indices]
    graphB = elegans['Agap'].toarray() [indices][:, indices]

    graphA2 = graphA.copy()

    avg_arr = []

    comm_len = [np.sum(elegans_labels_leiden[runs] == x) for x in range(0, max(elegans_labels_leiden[runs])+1)]

    for percentage in range(0,1,1):

          shuffled_perm = shuffle_percentage(list(range(0,279)), percentage)
          for num_of_seeds in [0,1,5,10,20,50,75,100,150,200]:
                print("-------------------------------------------------------")
                print("Current number of seeds:" + str(num_of_seeds))
                avg = 0
                seeds_in_comm2 = new_generate_array_comm(comm_len, num_of_seeds)
                for j in range(0, 10):
                      cum_comm = np.cumsum(comm_len)
                      cum_comm = [0] + cum_comm.tolist()
                      perm = np.array(range(0,cum_comm[-1]))


                      for i in range(0, len(seeds_in_comm2)):
                          slice_size = cum_comm[i+1] - (cum_comm[i] + seeds_in_comm2[i])
                          permuted_indices = np.random.permutation(slice_size)
                          shifted_indices = permuted_indices + (cum_comm[i] + seeds_in_comm2[i])
                          perm[(cum_comm[i]+seeds_in_comm2[i]):cum_comm[i+1]] = shifted_indices

                      graphA3 = graphA.copy()
                      graphA3[:] = graphA3[shuffled_perm][:, shuffled_perm]
                      graphA2[:] = graphA3[perm][:, perm]
                      #print(perm)
                      #print(len(comm_len))
                      #print(len(seeds_in_comm2))
                      corr = block_SGM(graphA2, graphB, comm_len, seeds_in_comm2, 20 , 1-10**(-6));

                      num_seeds = np.sum(seeds_in_comm2)
                      avg = avg+ ((np.sum(perm == corr)-num_seeds)/(279-num_seeds))
                      #print((np.sum(perm == corr)-num_seeds)/(279-num_seeds))

                avg_arr = avg_arr + [avg/10]
                #acc_arr_per = acc_arr_per + [avg/10]
                print("avg result for this number of seeds is "+str(avg/10))
                print("-------------------------------------------------------")
                if num_of_seeds not in acc_dict.keys():
                      acc_dict[num_of_seeds] = [avg/10]
                else:
                      acc_dict[num_of_seeds].append(avg/10)
          print("Current acc dict")
          print(acc_dict)


-------------------------------------------------------
Current number of seeds:0
avg result for this number of seeds is 0.15089605734767025
-------------------------------------------------------
-------------------------------------------------------
Current number of seeds:1
avg result for this number of seeds is 0.1589928057553957
-------------------------------------------------------
-------------------------------------------------------
Current number of seeds:5
avg result for this number of seeds is 0.14927007299270073
-------------------------------------------------------
-------------------------------------------------------
Current number of seeds:10
avg result for this number of seeds is 0.18178438661710034
-------------------------------------------------------
-------------------------------------------------------
Current number of seeds:20
avg result for this number of seeds is 0.18571428571428572
-------------------------------------------------------
--------------

In [ ]:
# leiden without singletons elegans experiments
elegans = scipy.io.loadmat('elegansGraph.mat')
graphA = elegans['Achem'].toarray()
graphB = elegans['Agap'].toarray()

all_leiden_elegans_here = np.load('100runs_leiden_elegans_with_singletons.npy', allow_pickle=True)
all_leiden_elegans_labels_no_singleton = []
for i in range(100):
    all_leiden_elegans_labels_no_singleton.append(leiden_communities_to_labels(merge_single_element_sublists(all_leiden_elegans_here[i])))
np.save('100runs_leiden_elegans_without_singletons_labels.npy', np.array(all_leiden_elegans_labels_no_singleton, dtype=object), allow_pickle=True)


In [ ]:
np.save('100runs_leiden_elegans_without_singletons_labels.npy', np.array(all_leiden_elegans_labels_no_singleton, dtype=object), allow_pickle=True)

elegans_labels_leiden = np.load('100runs_leiden_elegans_without_singletons_labels.npy', allow_pickle=True)

acc_dict = {}

for runs in range(100):

    indices = elegans_labels_leiden[runs].argsort()

    graphA = elegans['Achem'].toarray()[indices][:, indices]
    graphB = elegans['Agap'].toarray() [indices][:, indices]

    graphA2 = graphA.copy()

    avg_arr = []

    comm_len = [np.sum(elegans_labels_leiden[runs] == x) for x in range(0, max(elegans_labels_leiden[runs])+1)]

    for percentage in range(0,1,1):

          shuffled_perm = shuffle_percentage(list(range(0,279)), percentage)
          avg = 0
          for num_of_seeds in [0,1,5,10,20,50,75,100,150,200]:
                print("-------------------------------------------------------")
                print("Current number of seeds:" + str(num_of_seeds))
                avg = 0
                seeds_in_comm2 = new_generate_array_comm(comm_len, num_of_seeds)
                for j in range(0,3):
                      cum_comm = np.cumsum(comm_len)
                      cum_comm = [0] + cum_comm.tolist()
                      perm = np.array(range(0,cum_comm[-1]))


                      for i in range(0, len(seeds_in_comm2)):
                          slice_size = cum_comm[i+1] - (cum_comm[i] + seeds_in_comm2[i])
                          permuted_indices = np.random.permutation(slice_size)
                          shifted_indices = permuted_indices + (cum_comm[i] + seeds_in_comm2[i])
                          perm[(cum_comm[i]+seeds_in_comm2[i]):cum_comm[i+1]] = shifted_indices

                      graphA3 = graphA.copy()
                      graphA3[:] = graphA3[shuffled_perm][:, shuffled_perm]
                      graphA2[:] = graphA3[perm][:, perm]
                      #print(perm)
                      #print(len(comm_len))
                      #print(len(seeds_in_comm2))
                      corr = block_SGM(graphA2, graphB, comm_len, seeds_in_comm2, 20 , 1-10**(-6));

                      num_seeds = np.sum(seeds_in_comm2)
                      avg = avg+ ((np.sum(perm == corr)-num_seeds)/(279-num_seeds))
                      #print((np.sum(perm == corr)-num_seeds)/(279-num_seeds))

                avg_arr = avg_arr + [avg/10]
                #acc_arr_per = acc_arr_per + [avg/10]
                print("avg result for this number of seeds is "+str(avg/10))
                print("-------------------------------------------------------")
                if num_of_seeds not in acc_dict.keys():
                      acc_dict[num_of_seeds] = [avg/10]
                else:
                      acc_dict[num_of_seeds].append(avg/10)
          print("Current acc dict")
          print(acc_dict)


-------------------------------------------------------
Current number of seeds:0
avg result for this number of seeds is 0.023655913978494623
-------------------------------------------------------
-------------------------------------------------------
Current number of seeds:1
avg result for this number of seeds is 0.022302158273381296
-------------------------------------------------------
-------------------------------------------------------
Current number of seeds:5
avg result for this number of seeds is 0.022627737226277374
-------------------------------------------------------
-------------------------------------------------------
Current number of seeds:10
avg result for this number of seeds is 0.023791821561338293
-------------------------------------------------------
-------------------------------------------------------
Current number of seeds:20
avg result for this number of seeds is 0.020077220077220077
-------------------------------------------------------
--------

In [2]:
import numpy as np
all_leiden_elegans_labels_no_singleton_acc_dict = {0: [np.float64(0.023655913978494623), np.float64(0.020430107526881718), np.float64(0.024731182795698924), np.float64(0.024372759856630826), np.float64(0.017562724014336915), np.float64(0.02078853046594982), np.float64(0.017921146953405017), np.float64(0.02222222222222222), np.float64(0.024372759856630823), np.float64(0.02007168458781362), np.float64(0.016845878136200716), np.float64(0.02329749103942652), np.float64(0.023655913978494623), np.float64(0.025089605734767022), np.float64(0.019713261648745522), np.float64(0.026164874551971327), np.float64(0.023655913978494623), np.float64(0.024372759856630823), np.float64(0.024372759856630826), np.float64(0.024372759856630826), np.float64(0.01935483870967742), np.float64(0.020071684587813617), np.float64(0.025089605734767022), np.float64(0.01899641577060932), np.float64(0.026164874551971327), np.float64(0.015412186379928316), np.float64(0.02078853046594982), np.float64(0.025806451612903226), np.float64(0.023655913978494626), np.float64(0.02724014336917563), np.float64(0.03118279569892473), np.float64(0.02186379928315412), np.float64(0.025089605734767022), np.float64(0.026164874551971327), np.float64(0.02544802867383512), np.float64(0.024014336917562724), np.float64(0.026523297491039426), np.float64(0.02329749103942652), np.float64(0.01863799283154122), np.float64(0.02150537634408602), np.float64(0.022580645161290325), np.float64(0.026523297491039422), np.float64(0.024731182795698924), np.float64(0.02114695340501792), np.float64(0.02508960573476703), np.float64(0.025806451612903226), np.float64(0.022939068100358423), np.float64(0.025448028673835128), np.float64(0.022222222222222223), np.float64(0.025089605734767022), np.float64(0.024014336917562724), np.float64(0.02258064516129032), np.float64(0.02043010752688172), np.float64(0.025448028673835128), np.float64(0.022939068100358423), np.float64(0.02473118279569892), np.float64(0.02114695340501792), np.float64(0.024014336917562724), np.float64(0.02186379928315412), np.float64(0.023297491039426528), np.float64(0.026523297491039426), np.float64(0.016487455197132617), np.float64(0.020430107526881718), np.float64(0.026523297491039426), np.float64(0.01971326164874552), np.float64(0.026164874551971327), np.float64(0.017204301075268817), np.float64(0.02186379928315412), np.float64(0.025806451612903226), np.float64(0.03046594982078853), np.float64(0.017204301075268817), np.float64(0.024372759856630823), np.float64(0.024731182795698924), np.float64(0.017204301075268817), np.float64(0.02258064516129032), np.float64(0.027598566308243727), np.float64(0.022222222222222223), np.float64(0.024372759856630823), np.float64(0.022939068100358423), np.float64(0.02258064516129032), np.float64(0.024014336917562724), np.float64(0.026164874551971327), np.float64(0.02258064516129032), np.float64(0.017204301075268817), np.float64(0.021146953405017925), np.float64(0.02508960573476703), np.float64(0.016487455197132617), np.float64(0.026164874551971327), np.float64(0.01971326164874552), np.float64(0.02007168458781362), np.float64(0.020788530465949823), np.float64(0.023655913978494623), np.float64(0.01899641577060932), np.float64(0.02258064516129032), np.float64(0.022580645161290325), np.float64(0.024731182795698924), np.float64(0.02222222222222222), np.float64(0.02007168458781362), np.float64(0.018637992831541217), np.float64(0.016487455197132617)], 1: [np.float64(0.022302158273381296), np.float64(0.022302158273381296), np.float64(0.02302158273381295), np.float64(0.022302158273381296), np.float64(0.020143884892086333), np.float64(0.01870503597122302), np.float64(0.02050359712230216), np.float64(0.030215827338129497), np.float64(0.02410071942446043), np.float64(0.02158273381294964), np.float64(0.016906474820143885), np.float64(0.02446043165467626), np.float64(0.022661870503597123), np.float64(0.02266187050359712), np.float64(0.022661870503597123), np.float64(0.02446043165467626), np.float64(0.023741007194244608), np.float64(0.023381294964028777), np.float64(0.01906474820143885), np.float64(0.02302158273381295), np.float64(0.018345323741007193), np.float64(0.021582733812949638), np.float64(0.020143884892086333), np.float64(0.016187050359712234), np.float64(0.024820143884892086), np.float64(0.023021582733812947), np.float64(0.022302158273381296), np.float64(0.02122302158273381), np.float64(0.023741007194244605), np.float64(0.01870503597122302), np.float64(0.026618705035971225), np.float64(0.025179856115107913), np.float64(0.02410071942446043), np.float64(0.01870503597122302), np.float64(0.024820143884892086), np.float64(0.025179856115107913), np.float64(0.017625899280575542), np.float64(0.02302158273381295), np.float64(0.02194244604316547), np.float64(0.018345323741007193), np.float64(0.023381294964028777), np.float64(0.020143884892086333), np.float64(0.025899280575539568), np.float64(0.025899280575539568), np.float64(0.022302158273381296), np.float64(0.023741007194244608), np.float64(0.020863309352517984), np.float64(0.02302158273381295), np.float64(0.023741007194244605), np.float64(0.022302158273381296), np.float64(0.026618705035971225), np.float64(0.023381294964028777), np.float64(0.021582733812949638), np.float64(0.02446043165467626), np.float64(0.023741007194244605), np.float64(0.022302158273381296), np.float64(0.020863309352517987), np.float64(0.02050359712230216), np.float64(0.025539568345323744), np.float64(0.02410071942446043), np.float64(0.026618705035971225), np.float64(0.017985611510791366), np.float64(0.024460431654676255), np.float64(0.02733812949640288), np.float64(0.023381294964028777), np.float64(0.029496402877697843), np.float64(0.017985611510791366), np.float64(0.025539568345323737), np.float64(0.02194244604316547), np.float64(0.025899280575539568), np.float64(0.01870503597122302), np.float64(0.02697841726618705), np.float64(0.019064748201438848), np.float64(0.01906474820143885), np.float64(0.024820143884892086), np.float64(0.02410071942446043), np.float64(0.022302158273381296), np.float64(0.023741007194244605), np.float64(0.023381294964028777), np.float64(0.02410071942446043), np.float64(0.018345323741007193), np.float64(0.02194244604316547), np.float64(0.015107913669064749), np.float64(0.01942446043165468), np.float64(0.022302158273381296), np.float64(0.02302158273381295), np.float64(0.017625899280575542), np.float64(0.023021582733812947), np.float64(0.02194244604316547), np.float64(0.024820143884892086), np.float64(0.021223021582733814), np.float64(0.025539568345323744), np.float64(0.02194244604316547), np.float64(0.021223021582733814), np.float64(0.01906474820143885), np.float64(0.023381294964028777), np.float64(0.017985611510791366), np.float64(0.023381294964028777), np.float64(0.022302158273381296), np.float64(0.016906474820143885)], 5: [np.float64(0.022627737226277374), np.float64(0.021167883211678833), np.float64(0.020802919708029197), np.float64(0.02043795620437956), np.float64(0.020437956204379562), np.float64(0.020437956204379562), np.float64(0.02372262773722628), np.float64(0.022992700729927006), np.float64(0.01934306569343066), np.float64(0.024817518248175185), np.float64(0.02408759124087591), np.float64(0.02226277372262774), np.float64(0.02408759124087591), np.float64(0.025912408759124084), np.float64(0.021897810218978103), np.float64(0.02518248175182482), np.float64(0.017153284671532845), np.float64(0.02262773722627737), np.float64(0.021532846715328468), np.float64(0.022627737226277374), np.float64(0.021532846715328468), np.float64(0.024817518248175185), np.float64(0.020802919708029197), np.float64(0.020437956204379562), np.float64(0.023722627737226276), np.float64(0.01897810218978102), np.float64(0.022992700729927006), np.float64(0.022627737226277374), np.float64(0.028102189781021896), np.float64(0.02262773722627737), np.float64(0.029927007299270076), np.float64(0.022627737226277374), np.float64(0.02846715328467153), np.float64(0.020437956204379562), np.float64(0.02408759124087591), np.float64(0.02226277372262774), np.float64(0.02737226277372263), np.float64(0.01970802919708029), np.float64(0.017153284671532848), np.float64(0.022992700729927006), np.float64(0.020072992700729923), np.float64(0.026277372262773723), np.float64(0.022262773722627735), np.float64(0.019343065693430656), np.float64(0.01824817518248175), np.float64(0.022992700729927006), np.float64(0.0218978102189781), np.float64(0.023722627737226276), np.float64(0.023722627737226276), np.float64(0.019343065693430656), np.float64(0.025912408759124088), np.float64(0.020437956204379562), np.float64(0.021897810218978103), np.float64(0.023722627737226276), np.float64(0.025912408759124088), np.float64(0.0218978102189781), np.float64(0.020437956204379562), np.float64(0.019343065693430656), np.float64(0.021532846715328468), np.float64(0.022627737226277374), np.float64(0.02773722627737226), np.float64(0.01824817518248175), np.float64(0.027007299270072994), np.float64(0.02226277372262774), np.float64(0.025182481751824814), np.float64(0.02773722627737226), np.float64(0.021167883211678833), np.float64(0.01824817518248175), np.float64(0.024817518248175185), np.float64(0.03321167883211679), np.float64(0.021532846715328468), np.float64(0.01824817518248175), np.float64(0.02408759124087591), np.float64(0.020072992700729927), np.float64(0.023722627737226276), np.float64(0.026277372262773723), np.float64(0.019343065693430656), np.float64(0.020802919708029197), np.float64(0.016423357664233577), np.float64(0.022992700729927006), np.float64(0.024087591240875915), np.float64(0.021897810218978103), np.float64(0.020072992700729927), np.float64(0.0208029197080292), np.float64(0.020437956204379562), np.float64(0.025547445255474456), np.float64(0.021532846715328464), np.float64(0.0218978102189781), np.float64(0.025182481751824814), np.float64(0.024087591240875915), np.float64(0.020437956204379566), np.float64(0.02226277372262774), np.float64(0.024452554744525547), np.float64(0.01970802919708029), np.float64(0.019343065693430656), np.float64(0.024452554744525547), np.float64(0.01824817518248175), np.float64(0.016423357664233577), np.float64(0.02226277372262774), np.float64(0.022992700729927006)], 10: [np.float64(0.023791821561338293), np.float64(0.021561338289962827), np.float64(0.022304832713754646), np.float64(0.021561338289962824), np.float64(0.02267657992565056), np.float64(0.022304832713754646), np.float64(0.022304832713754646), np.float64(0.02267657992565056), np.float64(0.01933085501858736), np.float64(0.021933085501858733), np.float64(0.02639405204460966), np.float64(0.021561338289962827), np.float64(0.022676579925650555), np.float64(0.021189591078066915), np.float64(0.02527881040892193), np.float64(0.021561338289962827), np.float64(0.023420074349442377), np.float64(0.02304832713754647), np.float64(0.016728624535315983), np.float64(0.0275092936802974), np.float64(0.030111524163568777), np.float64(0.020446096654275093), np.float64(0.021561338289962827), np.float64(0.024163568773234202), np.float64(0.022304832713754646), np.float64(0.023791821561338293), np.float64(0.0275092936802974), np.float64(0.024163568773234202), np.float64(0.02230483271375465), np.float64(0.021561338289962827), np.float64(0.029368029739776952), np.float64(0.02490706319702602), np.float64(0.02453531598513011), np.float64(0.025650557620817842), np.float64(0.01895910780669145), np.float64(0.02342007434944238), np.float64(0.02453531598513011), np.float64(0.024163568773234202), np.float64(0.022304832713754646), np.float64(0.02379182156133829), np.float64(0.028996282527881046), np.float64(0.026765799256505574), np.float64(0.020446096654275096), np.float64(0.021189591078066915), np.float64(0.020817843866171006), np.float64(0.01895910780669145), np.float64(0.021561338289962827), np.float64(0.024163568773234202), np.float64(0.027881040892193308), np.float64(0.02639405204460966), np.float64(0.025650557620817842), np.float64(0.020817843866171006), np.float64(0.024163568773234202), np.float64(0.018587360594795536), np.float64(0.02379182156133829), np.float64(0.022304832713754646), np.float64(0.026022304832713755), np.float64(0.02379182156133829), np.float64(0.021561338289962824), np.float64(0.02304832713754647), np.float64(0.025650557620817842), np.float64(0.01635687732342007), np.float64(0.019330855018587362), np.float64(0.02490706319702602), np.float64(0.02453531598513011), np.float64(0.0275092936802974), np.float64(0.02453531598513011), np.float64(0.02453531598513011), np.float64(0.026394052044609668), np.float64(0.022304832713754646), np.float64(0.02267657992565056), np.float64(0.021933085501858733), np.float64(0.02379182156133829), np.float64(0.024163568773234202), np.float64(0.02379182156133829), np.float64(0.01895910780669145), np.float64(0.020446096654275093), np.float64(0.023048327137546468), np.float64(0.020446096654275093), np.float64(0.02453531598513011), np.float64(0.02453531598513011), np.float64(0.02230483271375465), np.float64(0.027881040892193308), np.float64(0.019702602230483268), np.float64(0.020446096654275093), np.float64(0.02490706319702602), np.float64(0.020817843866171006), np.float64(0.020446096654275093), np.float64(0.023048327137546468), np.float64(0.02342007434944238), np.float64(0.027137546468401486), np.float64(0.02490706319702602), np.float64(0.02453531598513011), np.float64(0.021933085501858733), np.float64(0.02304832713754647), np.float64(0.023420074349442377), np.float64(0.03159851301115242), np.float64(0.025650557620817842), np.float64(0.029368029739776952), np.float64(0.026022304832713748)], 20: [np.float64(0.020077220077220077), np.float64(0.017374517374517374), np.float64(0.021235521235521235), np.float64(0.018146718146718147), np.float64(0.025096525096525095), np.float64(0.02277992277992278), np.float64(0.021621621621621623), np.float64(0.01891891891891892), np.float64(0.018532818532818535), np.float64(0.021235521235521238), np.float64(0.028957528957528955), np.float64(0.016216216216216217), np.float64(0.01776061776061776), np.float64(0.016988416988416986), np.float64(0.015444015444015444), np.float64(0.014671814671814673), np.float64(0.021235521235521238), np.float64(0.018532818532818535), np.float64(0.020463320463320462), np.float64(0.017760617760617763), np.float64(0.02277992277992278), np.float64(0.02316602316602317), np.float64(0.021621621621621623), np.float64(0.02702702702702703), np.float64(0.018532818532818535), np.float64(0.025096525096525095), np.float64(0.024324324324324326), np.float64(0.020463320463320462), np.float64(0.019305019305019305), np.float64(0.01583011583011583), np.float64(0.03397683397683397), np.float64(0.013899613899613899), np.float64(0.020077220077220077), np.float64(0.018532818532818535), np.float64(0.01583011583011583), np.float64(0.02084942084942085), np.float64(0.027799227799227798), np.float64(0.017374517374517374), np.float64(0.02277992277992278), np.float64(0.02162162162162162), np.float64(0.019305019305019305), np.float64(0.017374517374517374), np.float64(0.019305019305019305), np.float64(0.023166023166023165), np.float64(0.01891891891891892), np.float64(0.021621621621621623), np.float64(0.018146718146718147), np.float64(0.018146718146718147), np.float64(0.016216216216216217), np.float64(0.020849420849420854), np.float64(0.018532818532818535), np.float64(0.022393822393822392), np.float64(0.033204633204633204), np.float64(0.018532818532818532), np.float64(0.022393822393822392), np.float64(0.020077220077220077), np.float64(0.025482625482625483), np.float64(0.019305019305019305), np.float64(0.02586872586872587), np.float64(0.021621621621621623), np.float64(0.032046332046332046), np.float64(0.027413127413127413), np.float64(0.020077220077220077), np.float64(0.02393822393822394), np.float64(0.016988416988416986), np.float64(0.0305019305019305), np.float64(0.022779922779922784), np.float64(0.021235521235521238), np.float64(0.016988416988416986), np.float64(0.022007722007722007), np.float64(0.02316602316602317), np.float64(0.018532818532818535), np.float64(0.016988416988416986), np.float64(0.02277992277992278), np.float64(0.013127413127413126), np.float64(0.018146718146718147), np.float64(0.01891891891891892), np.float64(0.019305019305019305), np.float64(0.023938223938223934), np.float64(0.021621621621621616), np.float64(0.016602316602316602), np.float64(0.016988416988416986), np.float64(0.02972972972972973), np.float64(0.024324324324324326), np.float64(0.02277992277992278), np.float64(0.02162162162162162), np.float64(0.025868725868725868), np.float64(0.01969111969111969), np.float64(0.023552123552123553), np.float64(0.023938223938223938), np.float64(0.02857142857142857), np.float64(0.02857142857142857), np.float64(0.027413127413127413), np.float64(0.020463320463320465), np.float64(0.015444015444015444), np.float64(0.01891891891891892), np.float64(0.0305019305019305), np.float64(0.026640926640926644), np.float64(0.026640926640926644), np.float64(0.026640926640926637)], 50: [np.float64(0.023580786026200874), np.float64(0.020087336244541485), np.float64(0.026200873362445413), np.float64(0.02096069868995633), np.float64(0.034934497816593885), np.float64(0.03668122270742358), np.float64(0.03537117903930131), np.float64(0.02445414847161572), np.float64(0.02314410480349345), np.float64(0.02096069868995633), np.float64(0.03275109170305677), np.float64(0.02576419213973799), np.float64(0.024017467248908297), np.float64(0.02183406113537118), np.float64(0.024017467248908297), np.float64(0.01834061135371179), np.float64(0.023144104803493454), np.float64(0.01921397379912664), np.float64(0.02183406113537118), np.float64(0.024017467248908297), np.float64(0.029257641921397383), np.float64(0.024017467248908297), np.float64(0.019650655021834062), np.float64(0.03406113537117904), np.float64(0.01965065502183406), np.float64(0.03275109170305677), np.float64(0.03056768558951965), np.float64(0.01921397379912664), np.float64(0.022270742358078605), np.float64(0.01965065502183406), np.float64(0.033624454148471615), np.float64(0.020087336244541485), np.float64(0.023144104803493448), np.float64(0.01834061135371179), np.float64(0.02576419213973799), np.float64(0.021397379912663755), np.float64(0.03449781659388646), np.float64(0.022270742358078605), np.float64(0.029694323144104806), np.float64(0.03973799126637555), np.float64(0.020087336244541485), np.float64(0.017467248908296946), np.float64(0.02183406113537118), np.float64(0.037554585152838424), np.float64(0.023580786026200874), np.float64(0.022707423580786028), np.float64(0.01965065502183406), np.float64(0.023144104803493448), np.float64(0.01834061135371179), np.float64(0.023580786026200874), np.float64(0.02052401746724891), np.float64(0.027510917030567683), np.float64(0.03406113537117904), np.float64(0.024017467248908297), np.float64(0.020524017467248912), np.float64(0.022270742358078598), np.float64(0.026200873362445413), np.float64(0.020960698689956335), np.float64(0.03056768558951965), np.float64(0.024017467248908297), np.float64(0.027510917030567683), np.float64(0.03056768558951965), np.float64(0.02183406113537118), np.float64(0.02052401746724891), np.float64(0.02445414847161572), np.float64(0.029694323144104806), np.float64(0.03013100436681223), np.float64(0.02096069868995633), np.float64(0.02532751091703057), np.float64(0.030131004366812226), np.float64(0.03973799126637554), np.float64(0.014410480349344978), np.float64(0.0222707423580786), np.float64(0.039301310043668124), np.float64(0.024890829694323147), np.float64(0.02183406113537118), np.float64(0.01834061135371179), np.float64(0.02445414847161572), np.float64(0.028384279475982533), np.float64(0.022707423580786024), np.float64(0.02096069868995633), np.float64(0.021397379912663758), np.float64(0.033624454148471615), np.float64(0.0406113537117904), np.float64(0.032314410480349345), np.float64(0.022707423580786028), np.float64(0.04192139737991266), np.float64(0.026200873362445413), np.float64(0.03318777292576419), np.float64(0.031877729257641915), np.float64(0.03493449781659389), np.float64(0.03187772925764192), np.float64(0.029257641921397383), np.float64(0.027510917030567683), np.float64(0.0222707423580786), np.float64(0.02445414847161572), np.float64(0.029257641921397383), np.float64(0.032314410480349345), np.float64(0.04148471615720524), np.float64(0.029257641921397383)], 75: [np.float64(0.025980392156862746), np.float64(0.030392156862745094), np.float64(0.03137254901960784), np.float64(0.029411764705882353), np.float64(0.043137254901960784), np.float64(0.040196078431372545), np.float64(0.039215686274509796), np.float64(0.02401960784313726), np.float64(0.02892156862745098), np.float64(0.026960784313725488), np.float64(0.04068627450980392), np.float64(0.027450980392156866), np.float64(0.02892156862745098), np.float64(0.028431372549019607), np.float64(0.029901960784313723), np.float64(0.029411764705882353), np.float64(0.026960784313725488), np.float64(0.028921568627450978), np.float64(0.029411764705882353), np.float64(0.02303921568627451), np.float64(0.043627450980392155), np.float64(0.030882352941176472), np.float64(0.028431372549019607), np.float64(0.043137254901960784), np.float64(0.030882352941176472), np.float64(0.03970588235294118), np.float64(0.04656862745098039), np.float64(0.027941176470588237), np.float64(0.027450980392156866), np.float64(0.026960784313725488), np.float64(0.030392156862745094), np.float64(0.023529411764705882), np.float64(0.02941176470588235), np.float64(0.02303921568627451), np.float64(0.025490196078431372), np.float64(0.02254901960784314), np.float64(0.0303921568627451), np.float64(0.025980392156862746), np.float64(0.03529411764705882), np.float64(0.039705882352941174), np.float64(0.030882352941176472), np.float64(0.027941176470588237), np.float64(0.02450980392156863), np.float64(0.03725490196078431), np.float64(0.02450980392156863), np.float64(0.024019607843137256), np.float64(0.025), np.float64(0.029901960784313723), np.float64(0.028431372549019607), np.float64(0.026470588235294117), np.float64(0.02941176470588235), np.float64(0.04068627450980392), np.float64(0.04509803921568627), np.float64(0.029411764705882353), np.float64(0.026960784313725488), np.float64(0.029901960784313723), np.float64(0.0446078431372549), np.float64(0.0303921568627451), np.float64(0.026960784313725488), np.float64(0.02401960784313726), np.float64(0.030882352941176472), np.float64(0.028431372549019607), np.float64(0.029901960784313723), np.float64(0.027941176470588237), np.float64(0.022058823529411763), np.float64(0.036274509803921565), np.float64(0.0446078431372549), np.float64(0.025490196078431372), np.float64(0.028431372549019607), np.float64(0.03333333333333333), np.float64(0.04068627450980392), np.float64(0.026960784313725488), np.float64(0.02941176470588235), np.float64(0.039215686274509796), np.float64(0.026470588235294117), np.float64(0.02843137254901961), np.float64(0.022058823529411766), np.float64(0.02303921568627451), np.float64(0.032352941176470584), np.float64(0.025980392156862746), np.float64(0.025490196078431372), np.float64(0.02450980392156863), np.float64(0.044607843137254896), np.float64(0.03676470588235294), np.float64(0.037745098039215684), np.float64(0.025), np.float64(0.03970588235294118), np.float64(0.029411764705882353), np.float64(0.039705882352941174), np.float64(0.04019607843137255), np.float64(0.042156862745098035), np.float64(0.03431372549019608), np.float64(0.036274509803921565), np.float64(0.03480392156862745), np.float64(0.02450980392156863), np.float64(0.02401960784313726), np.float64(0.04166666666666667), np.float64(0.03382352941176471), np.float64(0.03431372549019608), np.float64(0.04264705882352941)], 100: [np.float64(0.03128491620111732), np.float64(0.037430167597765365), np.float64(0.03240223463687151), np.float64(0.0335195530726257), np.float64(0.04636871508379888), np.float64(0.04692737430167598), np.float64(0.05083798882681565), np.float64(0.03240223463687151), np.float64(0.031843575418994415), np.float64(0.03519553072625699), np.float64(0.05418994413407822), np.float64(0.03687150837988827), np.float64(0.03016759776536313), np.float64(0.03128491620111732), np.float64(0.03519553072625699), np.float64(0.0329608938547486), np.float64(0.030726256983240226), np.float64(0.0340782122905028), np.float64(0.03407821229050279), np.float64(0.0335195530726257), np.float64(0.051955307262569826), np.float64(0.03240223463687151), np.float64(0.03463687150837989), np.float64(0.053072625698324015), np.float64(0.03240223463687151), np.float64(0.05418994413407822), np.float64(0.05027932960893855), np.float64(0.03519553072625699), np.float64(0.0335195530726257), np.float64(0.0329608938547486), np.float64(0.04581005586592179), np.float64(0.038547486033519554), np.float64(0.034636871508379886), np.float64(0.03407821229050279), np.float64(0.0335195530726257), np.float64(0.03407821229050279), np.float64(0.037430167597765365), np.float64(0.03296089385474861), np.float64(0.036312849162011177), np.float64(0.049162011173184354), np.float64(0.03184357541899442), np.float64(0.03463687150837989), np.float64(0.027374301675977653), np.float64(0.04692737430167598), np.float64(0.03296089385474861), np.float64(0.03240223463687151), np.float64(0.03240223463687151), np.float64(0.03687150837988827), np.float64(0.0340782122905028), np.float64(0.034636871508379886), np.float64(0.03240223463687151), np.float64(0.03910614525139665), np.float64(0.051396648044692725), np.float64(0.03575418994413408), np.float64(0.03128491620111732), np.float64(0.036312849162011177), np.float64(0.04581005586592179), np.float64(0.03910614525139665), np.float64(0.041340782122905026), np.float64(0.03240223463687151), np.float64(0.031843575418994415), np.float64(0.03575418994413408), np.float64(0.03519553072625698), np.float64(0.03128491620111732), np.float64(0.03128491620111732), np.float64(0.03575418994413408), np.float64(0.041340782122905026), np.float64(0.03798882681564246), np.float64(0.03240223463687151), np.float64(0.04804469273743017), np.float64(0.05251396648044693), np.float64(0.0340782122905028), np.float64(0.029050279329608943), np.float64(0.05083798882681565), np.float64(0.031843575418994415), np.float64(0.03687150837988827), np.float64(0.03128491620111732), np.float64(0.03575418994413408), np.float64(0.0329608938547486), np.float64(0.0335195530726257), np.float64(0.03240223463687151), np.float64(0.037430167597765365), np.float64(0.04748603351955307), np.float64(0.05139664804469274), np.float64(0.034636871508379886), np.float64(0.036312849162011177), np.float64(0.05083798882681564), np.float64(0.031843575418994415), np.float64(0.049720670391061456), np.float64(0.051955307262569826), np.float64(0.04581005586592179), np.float64(0.04860335195530726), np.float64(0.04860335195530726), np.float64(0.034636871508379886), np.float64(0.031843575418994415), np.float64(0.03798882681564246), np.float64(0.04860335195530726), np.float64(0.038547486033519554), np.float64(0.046927374301675984), np.float64(0.05251396648044693)], 150: [np.float64(0.056589147286821705), np.float64(0.061240310077519386), np.float64(0.061240310077519386), np.float64(0.060465116279069774), np.float64(0.06434108527131784), np.float64(0.06201550387596899), np.float64(0.06976744186046512), np.float64(0.058139534883720936), np.float64(0.06666666666666668), np.float64(0.05968992248062016), np.float64(0.06589147286821706), np.float64(0.05968992248062015), np.float64(0.06356589147286822), np.float64(0.05968992248062015), np.float64(0.06201550387596899), np.float64(0.05968992248062015), np.float64(0.060465116279069774), np.float64(0.06046511627906977), np.float64(0.05348837209302326), np.float64(0.06279069767441861), np.float64(0.05891472868217055), np.float64(0.05891472868217055), np.float64(0.06201550387596899), np.float64(0.06899224806201551), np.float64(0.06511627906976744), np.float64(0.06666666666666668), np.float64(0.06976744186046512), np.float64(0.05968992248062016), np.float64(0.06666666666666667), np.float64(0.058139534883720936), np.float64(0.05891472868217055), np.float64(0.05968992248062016), np.float64(0.06279069767441861), np.float64(0.06201550387596899), np.float64(0.06279069767441861), np.float64(0.06201550387596899), np.float64(0.06279069767441861), np.float64(0.05891472868217055), np.float64(0.05348837209302326), np.float64(0.06124031007751938), np.float64(0.06201550387596899), np.float64(0.05736434108527132), np.float64(0.05426356589147287), np.float64(0.06821705426356589), np.float64(0.06046511627906977), np.float64(0.05193798449612404), np.float64(0.06201550387596899), np.float64(0.06589147286821706), np.float64(0.05968992248062015), np.float64(0.05891472868217055), np.float64(0.05891472868217055), np.float64(0.060465116279069774), np.float64(0.07131782945736434), np.float64(0.06511627906976744), np.float64(0.05736434108527132), np.float64(0.06279069767441861), np.float64(0.06511627906976744), np.float64(0.06511627906976744), np.float64(0.058914728682170535), np.float64(0.06046511627906977), np.float64(0.05736434108527132), np.float64(0.08372093023255814), np.float64(0.05736434108527132), np.float64(0.05426356589147287), np.float64(0.05968992248062016), np.float64(0.06356589147286822), np.float64(0.06201550387596899), np.float64(0.06356589147286822), np.float64(0.06201550387596899), np.float64(0.06434108527131784), np.float64(0.06356589147286822), np.float64(0.06511627906976744), np.float64(0.06046511627906977), np.float64(0.06899224806201551), np.float64(0.05891472868217055), np.float64(0.05968992248062016), np.float64(0.06434108527131784), np.float64(0.058914728682170535), np.float64(0.05581395348837209), np.float64(0.05426356589147287), np.float64(0.06201550387596899), np.float64(0.06356589147286822), np.float64(0.06899224806201551), np.float64(0.0682170542635659), np.float64(0.05426356589147287), np.float64(0.06511627906976744), np.float64(0.05968992248062015), np.float64(0.05891472868217055), np.float64(0.06434108527131784), np.float64(0.06589147286821706), np.float64(0.06589147286821706), np.float64(0.047286821705426356), np.float64(0.050387596899224806), np.float64(0.056589147286821705), np.float64(0.06511627906976744), np.float64(0.06046511627906977), np.float64(0.06589147286821706), np.float64(0.060465116279069774), np.float64(0.06511627906976744), np.float64(0.06434108527131784)], 200: [np.float64(0.07721518987341772), np.float64(0.08607594936708861), np.float64(0.07974683544303797), np.float64(0.07721518987341772), np.float64(0.08354430379746836), np.float64(0.08860759493670886), np.float64(0.07848101265822785), np.float64(0.08354430379746834), np.float64(0.08481012658227848), np.float64(0.09240506329113925), np.float64(0.08101265822784812), np.float64(0.07848101265822785), np.float64(0.07468354430379746), np.float64(0.07594936708860758), np.float64(0.0860759493670886), np.float64(0.07848101265822785), np.float64(0.08101265822784809), np.float64(0.08860759493670886), np.float64(0.07848101265822785), np.float64(0.08227848101265822), np.float64(0.0860759493670886), np.float64(0.0810126582278481), np.float64(0.08227848101265822), np.float64(0.08860759493670886), np.float64(0.0810126582278481), np.float64(0.07721518987341772), np.float64(0.08227848101265822), np.float64(0.07721518987341772), np.float64(0.07974683544303798), np.float64(0.07468354430379746), np.float64(0.0759493670886076), np.float64(0.08481012658227849), np.float64(0.0759493670886076), np.float64(0.07468354430379746), np.float64(0.07974683544303797), np.float64(0.07468354430379746), np.float64(0.08860759493670886), np.float64(0.08227848101265822), np.float64(0.06835443037974684), np.float64(0.08481012658227849), np.float64(0.08101265822784812), np.float64(0.08354430379746836), np.float64(0.0759493670886076), np.float64(0.08860759493670886), np.float64(0.07848101265822785), np.float64(0.07341772151898734), np.float64(0.07974683544303797), np.float64(0.08734177215189873), np.float64(0.07848101265822785), np.float64(0.08354430379746836), np.float64(0.08481012658227848), np.float64(0.11645569620253164), np.float64(0.08227848101265822), np.float64(0.08227848101265822), np.float64(0.07594936708860758), np.float64(0.08227848101265822), np.float64(0.09493670886075949), np.float64(0.07848101265822785), np.float64(0.09620253164556962), np.float64(0.07974683544303798), np.float64(0.07848101265822785), np.float64(0.06455696202531645), np.float64(0.08481012658227848), np.float64(0.0759493670886076), np.float64(0.08227848101265824), np.float64(0.1), np.float64(0.08227848101265824), np.float64(0.08734177215189873), np.float64(0.08227848101265822), np.float64(0.09746835443037974), np.float64(0.08354430379746836), np.float64(0.08227848101265822), np.float64(0.07721518987341772), np.float64(0.08227848101265824), np.float64(0.0759493670886076), np.float64(0.08101265822784809), np.float64(0.07721518987341772), np.float64(0.07974683544303798), np.float64(0.07721518987341772), np.float64(0.0810126582278481), np.float64(0.08481012658227848), np.float64(0.0860759493670886), np.float64(0.08481012658227848), np.float64(0.08227848101265822), np.float64(0.06329113924050633), np.float64(0.08987341772151898), np.float64(0.0810126582278481), np.float64(0.07721518987341772), np.float64(0.08354430379746836), np.float64(0.08354430379746834), np.float64(0.07848101265822785), np.float64(0.0620253164556962), np.float64(0.06455696202531645), np.float64(0.07215189873417722), np.float64(0.07974683544303797), np.float64(0.0860759493670886), np.float64(0.07848101265822785), np.float64(0.0810126582278481), np.float64(0.09620253164556962), np.float64(0.08227848101265822)]}

In [3]:
all_leiden_elegans_labels_no_singleton_acc_dict
acc_no_singleton_avg_dict = {
    k: float(np.mean(v) * (10 / 3))
    for k, v in all_leiden_elegans_labels_no_singleton_acc_dict.items()
}
acc_no_singleton_avg_dict

{0: 0.07574671445639188,
 1: 0.07452038369304557,
 5: 0.07475669099756692,
 10: 0.07804213135068155,
 20: 0.07166023166023167,
 50: 0.08748180494905385,
 75: 0.10480392156862743,
 100: 0.12783985102420858,
 150: 0.20547803617571062,
 200: 0.2716877637130802}

In [8]:
all_leiden_elegans_labels_with_singleton_acc_dict= {0: [np.float64(0.15089605734767025), np.float64(0.17562724014336917), np.float64(0.17347670250896058), np.float64(0.16702508960573478), np.float64(0.16630824372759853), np.float64(0.16129032258064518), np.float64(0.16379928315412187), np.float64(0.16559139784946236), np.float64(0.1659498207885305), np.float64(0.17060931899641577), np.float64(0.15519713261648743), np.float64(0.16810035842293908), np.float64(0.16272401433691758), np.float64(0.14372759856630823), np.float64(0.16845878136200715), np.float64(0.15734767025089608), np.float64(0.15663082437275985), np.float64(0.16881720430107525), np.float64(0.17634408602150536), np.float64(0.16021505376344086), np.float64(0.18566308243727597), np.float64(0.1695340501792115), np.float64(0.1706093189964158), np.float64(0.16523297491039426), np.float64(0.167741935483871), np.float64(0.17347670250896058), np.float64(0.17992831541218637), np.float64(0.17132616487455196), np.float64(0.15197132616487455), np.float64(0.15448028673835126), np.float64(0.17240143369175626), np.float64(0.16917562724014334), np.float64(0.16917562724014337), np.float64(0.15125448028673835), np.float64(0.16308243727598565), np.float64(0.16881720430107525), np.float64(0.1659498207885305), np.float64(0.16523297491039426), np.float64(0.16200716845878138), np.float64(0.16057347670250896), np.float64(0.16666666666666669), np.float64(0.16523297491039424), np.float64(0.15089605734767023), np.float64(0.17240143369175628), np.float64(0.16810035842293908), np.float64(0.17204301075268819), np.float64(0.1544802867383513), np.float64(0.16559139784946236), np.float64(0.16738351254480288), np.float64(0.1648745519713262), np.float64(0.17813620071684588), np.float64(0.15698924731182795), np.float64(0.17168458781362012), np.float64(0.16630824372759856), np.float64(0.17025089605734764), np.float64(0.17777777777777778), np.float64(0.14767025089605734), np.float64(0.1727598566308244), np.float64(0.17383512544802868), np.float64(0.19390681003584231), np.float64(0.15017921146953403), np.float64(0.16272401433691758), np.float64(0.16451612903225807), np.float64(0.15304659498207884), np.float64(0.17670250896057346), np.float64(0.16308243727598568), np.float64(0.16702508960573478), np.float64(0.17204301075268819), np.float64(0.15806451612903227), np.float64(0.17060931899641577), np.float64(0.16953405017921147), np.float64(0.16989247311827957), np.float64(0.16093189964157706), np.float64(0.1530465949820789), np.float64(0.16559139784946236), np.float64(0.16774193548387095), np.float64(0.15197132616487455), np.float64(0.16953405017921147), np.float64(0.14946236559139786), np.float64(0.15125448028673835), np.float64(0.14659498207885305), np.float64(0.15770609318996415), np.float64(0.15985663082437274), np.float64(0.16702508960573478), np.float64(0.16989247311827957), np.float64(0.17419354838709677), np.float64(0.15053763440860216), np.float64(0.15949820788530464), np.float64(0.16272401433691758), np.float64(0.14982078853046596)], 1: [np.float64(0.1589928057553957), np.float64(0.16618705035971224), np.float64(0.17517985611510792), np.float64(0.158273381294964), np.float64(0.17014388489208634), np.float64(0.16223021582733815), np.float64(0.16115107913669066), np.float64(0.17158273381294964), np.float64(0.15611510791366906), np.float64(0.16870503597122302), np.float64(0.1607913669064748), np.float64(0.1669064748201439), np.float64(0.16762589928057553), np.float64(0.15143884892086332), np.float64(0.1647482014388489), np.float64(0.15683453237410072), np.float64(0.1489208633093525), np.float64(0.1762589928057554), np.float64(0.16115107913669063), np.float64(0.1589928057553957), np.float64(0.1852517985611511), np.float64(0.1597122302158273), np.float64(0.16834532374100716), np.float64(0.16366906474820142), np.float64(0.16223021582733815), np.float64(0.16402877697841728), np.float64(0.16115107913669063), np.float64(0.16726618705035973), np.float64(0.16115107913669066), np.float64(0.1593525179856115), np.float64(0.16510791366906474), np.float64(0.16798561151079136), np.float64(0.17122302158273378), np.float64(0.1568345323741007), np.float64(0.16402877697841728), np.float64(0.17410071942446043), np.float64(0.16798561151079136), np.float64(0.1697841726618705), np.float64(0.16798561151079136), np.float64(0.1618705035971223), np.float64(0.1618705035971223), np.float64(0.168705035971223), np.float64(0.14999999999999997), np.float64(0.16187050359712232), np.float64(0.1589928057553957), np.float64(0.17014388489208632), np.float64(0.15647482014388486), np.float64(0.16834532374100722), np.float64(0.16402877697841725), np.float64(0.16366906474820142), np.float64(0.16798561151079133), np.float64(0.1593525179856115), np.float64(0.1647482014388489), np.float64(0.16366906474820145), np.float64(0.16151079136690646), np.float64(0.18345323741007197), np.float64(0.15539568345323743), np.float64(0.1629496402877698), np.float64(0.1618705035971223), np.float64(0.1679856115107914), np.float64(0.15395683453237408), np.float64(0.15863309352517987), np.float64(0.17086330935251798), np.float64(0.15791366906474819), np.float64(0.16474820143884888), np.float64(0.1647482014388489), np.float64(0.16258992805755393), np.float64(0.16294964028776976), np.float64(0.1564748201438849), np.float64(0.16223021582733813), np.float64(0.16330935251798565), np.float64(0.1672661870503597), np.float64(0.16187050359712232), np.float64(0.14820143884892087), np.float64(0.15827338129496402), np.float64(0.15611510791366906), np.float64(0.15755395683453238), np.float64(0.16726618705035973), np.float64(0.15575539568345326), np.float64(0.15323741007194244), np.float64(0.1485611510791367), np.float64(0.1564748201438849), np.float64(0.16906474820143885), np.float64(0.16510791366906474), np.float64(0.15899280575539568), np.float64(0.16798561151079133), np.float64(0.15611510791366903), np.float64(0.16223021582733815), np.float64(0.1629496402877698), np.float64(0.15395683453237408)], 5: [np.float64(0.14927007299270073), np.float64(0.17043795620437957), np.float64(0.1675182481751825), np.float64(0.1594890510948905), np.float64(0.1697080291970803), np.float64(0.17189781021897813), np.float64(0.17153284671532848), np.float64(0.1624087591240876), np.float64(0.16496350364963502), np.float64(0.1718978102189781), np.float64(0.15839416058394162), np.float64(0.16642335766423358), np.float64(0.16204379562043797), np.float64(0.15802919708029198), np.float64(0.16386861313868614), np.float64(0.1562043795620438), np.float64(0.1583941605839416), np.float64(0.1616788321167883), np.float64(0.1678832116788321), np.float64(0.1645985401459854), np.float64(0.19014598540145985), np.float64(0.168978102189781), np.float64(0.16496350364963502), np.float64(0.16313868613138688), np.float64(0.16788321167883213), np.float64(0.17116788321167883), np.float64(0.17189781021897813), np.float64(0.16715328467153284), np.float64(0.16240875912408761), np.float64(0.1642335766423358), np.float64(0.16897810218978102), np.float64(0.1686131386861314), np.float64(0.1627737226277372), np.float64(0.1605839416058394), np.float64(0.16131386861313868), np.float64(0.17372262773722627), np.float64(0.17007299270072995), np.float64(0.16423357664233576), np.float64(0.1678832116788321), np.float64(0.16970802919708028), np.float64(0.17518248175182483), np.float64(0.16715328467153284), np.float64(0.15693430656934307), np.float64(0.16094890510948906), np.float64(0.16058394160583944), np.float64(0.17518248175182483), np.float64(0.1594890510948905), np.float64(0.16423357664233576), np.float64(0.16715328467153284), np.float64(0.16897810218978104), np.float64(0.172992700729927), np.float64(0.158029197080292), np.float64(0.16934306569343066), np.float64(0.17043795620437957), np.float64(0.16934306569343066), np.float64(0.1846715328467153), np.float64(0.15729927007299271), np.float64(0.1700729927007299), np.float64(0.16131386861313868), np.float64(0.17846715328467153), np.float64(0.1591240875912409), np.float64(0.16678832116788322), np.float64(0.16313868613138688), np.float64(0.1583941605839416), np.float64(0.1653284671532847), np.float64(0.1656934306569343), np.float64(0.1718978102189781), np.float64(0.16569343065693432), np.float64(0.17262773722627736), np.float64(0.1635036496350365), np.float64(0.1686131386861314), np.float64(0.16751824817518252), np.float64(0.16204379562043794), np.float64(0.16131386861313868), np.float64(0.1711678832116788), np.float64(0.16204379562043797), np.float64(0.15620437956204378), np.float64(0.16642335766423358), np.float64(0.15875912408759124), np.float64(0.15839416058394162), np.float64(0.16021897810218977), np.float64(0.181021897810219), np.float64(0.1740875912408759), np.float64(0.16934306569343066), np.float64(0.1645985401459854), np.float64(0.1708029197080292), np.float64(0.15693430656934307), np.float64(0.16386861313868611), np.float64(0.1795620437956204), np.float64(0.15474452554744528)], 10: [np.float64(0.18178438661710034), np.float64(0.17546468401486986), np.float64(0.1687732342007435), np.float64(0.16877323420074347), np.float64(0.17174721189591077), np.float64(0.16617100371747212), np.float64(0.17769516728624535), np.float64(0.17137546468401485), np.float64(0.1758364312267658), np.float64(0.1691449814126394), np.float64(0.1866171003717472), np.float64(0.17843866171003717), np.float64(0.17137546468401488), np.float64(0.17769516728624535), np.float64(0.16765799256505579), np.float64(0.1695167286245353), np.float64(0.1892193308550186), np.float64(0.16765799256505576), np.float64(0.16505576208178438), np.float64(0.17397769516728626), np.float64(0.2007434944237918), np.float64(0.17434944237918215), np.float64(0.1657992565055762), np.float64(0.16765799256505579), np.float64(0.17063197026022306), np.float64(0.17100371747211898), np.float64(0.1799256505576208), np.float64(0.18029739776951673), np.float64(0.1609665427509294), np.float64(0.16988847583643124), np.float64(0.16765799256505576), np.float64(0.16951672862453532), np.float64(0.16840148698884758), np.float64(0.16654275092936804), np.float64(0.16988847583643124), np.float64(0.1762081784386617), np.float64(0.17286245353159851), np.float64(0.17286245353159851), np.float64(0.1654275092936803), np.float64(0.17137546468401488), np.float64(0.1765799256505576), np.float64(0.17174721189591077), np.float64(0.18550185873605946), np.float64(0.17509293680297397), np.float64(0.17026022304832716), np.float64(0.17137546468401488), np.float64(0.17732342007434942), np.float64(0.17063197026022306), np.float64(0.17695167286245356), np.float64(0.18401486988847582), np.float64(0.17843866171003717), np.float64(0.15799256505576206), np.float64(0.17063197026022306), np.float64(0.17063197026022306), np.float64(0.1817843866171004), np.float64(0.18587360594795538), np.float64(0.1795539033457249), np.float64(0.16802973977695168), np.float64(0.1795539033457249), np.float64(0.16877323420074347), np.float64(0.1691449814126394), np.float64(0.17286245353159851), np.float64(0.1643122676579926), np.float64(0.17434944237918212), np.float64(0.17397769516728628), np.float64(0.17026022304832714), np.float64(0.1698884758364312), np.float64(0.1661710037174721), np.float64(0.16356877323420074), np.float64(0.1732342007434944), np.float64(0.17174721189591077), np.float64(0.17100371747211898), np.float64(0.1758364312267658), np.float64(0.16951672862453532), np.float64(0.16059479553903344), np.float64(0.1724907063197026), np.float64(0.1661710037174721), np.float64(0.16542750929368027), np.float64(0.17732342007434945), np.float64(0.18029739776951673), np.float64(0.18178438661710036), np.float64(0.1821561338289963), np.float64(0.18773234200743497), np.float64(0.16691449814126394), np.float64(0.17620817843866168), np.float64(0.1724907063197026), np.float64(0.18513011152416356), np.float64(0.16765799256505579), np.float64(0.1851301115241636), np.float64(0.18066914498141262)], 20: [np.float64(0.18571428571428572), np.float64(0.155984555984556), np.float64(0.15250965250965248), np.float64(0.15598455598455596), np.float64(0.15366795366795366), np.float64(0.16061776061776062), np.float64(0.15868725868725866), np.float64(0.16216216216216214), np.float64(0.15984555984555987), np.float64(0.15250965250965248), np.float64(0.19073359073359075), np.float64(0.1548262548262548), np.float64(0.1528957528957529), np.float64(0.1656370656370656), np.float64(0.1555984555984556), np.float64(0.167953667953668), np.float64(0.18803088803088805), np.float64(0.1548262548262548), np.float64(0.15907335907335907), np.float64(0.15791505791505792), np.float64(0.18532818532818535), np.float64(0.1552123552123552), np.float64(0.1528957528957529), np.float64(0.1633204633204633), np.float64(0.1563706563706564), np.float64(0.15250965250965248), np.float64(0.20810810810810812), np.float64(0.15791505791505792), np.float64(0.17220077220077223), np.float64(0.16988416988416988), np.float64(0.15752895752895751), np.float64(0.15830115830115835), np.float64(0.15830115830115832), np.float64(0.16486486486486485), np.float64(0.15173745173745173), np.float64(0.1552123552123552), np.float64(0.16100386100386102), np.float64(0.15830115830115826), np.float64(0.15250965250965248), np.float64(0.15366795366795363), np.float64(0.15135135135135133), np.float64(0.17142857142857143), np.float64(0.17799227799227801), np.float64(0.15366795366795363), np.float64(0.15984555984555984), np.float64(0.15714285714285714), np.float64(0.1845559845559846), np.float64(0.14710424710424708), np.float64(0.18030888030888031), np.float64(0.15637065637065634), np.float64(0.2034749034749035), np.float64(0.16833976833976833), np.float64(0.16602316602316605), np.float64(0.16254826254826255), np.float64(0.1583011583011583), np.float64(0.18957528957528963), np.float64(0.19073359073359075), np.float64(0.15405405405405403), np.float64(0.1586872586872587), np.float64(0.19073359073359075), np.float64(0.1664092664092664), np.float64(0.15598455598455596), np.float64(0.15173745173745173), np.float64(0.16795366795366795), np.float64(0.15366795366795366), np.float64(0.16254826254826255), np.float64(0.15598455598455593), np.float64(0.1579150579150579), np.float64(0.17490347490347494), np.float64(0.14826254826254825), np.float64(0.16254826254826255), np.float64(0.155984555984556), np.float64(0.18301158301158305), np.float64(0.16718146718146717), np.float64(0.17644787644787646), np.float64(0.15598455598455596), np.float64(0.16563706563706562), np.float64(0.15173745173745173), np.float64(0.18339768339768342), np.float64(0.18841698841698845), np.float64(0.18108108108108112), np.float64(0.1818532818532819), np.float64(0.17528957528957528), np.float64(0.18030888030888034), np.float64(0.15675675675675674), np.float64(0.15289575289575286), np.float64(0.1814671814671815), np.float64(0.18339768339768342), np.float64(0.18957528957528963), np.float64(0.18725868725868727)], 50: [np.float64(0.21091703056768557), np.float64(0.1794759825327511), np.float64(0.19257641921397378), np.float64(0.18646288209606987), np.float64(0.18122270742358076), np.float64(0.17772925764192143), np.float64(0.17860262008733624), np.float64(0.18078602620087333), np.float64(0.17772925764192143), np.float64(0.18733624454148473), np.float64(0.2091703056768559), np.float64(0.18733624454148473), np.float64(0.18253275109170305), np.float64(0.22008733624454152), np.float64(0.1890829694323144), np.float64(0.22139737991266376), np.float64(0.21484716157205241), np.float64(0.18034934497816596), np.float64(0.1790393013100437), np.float64(0.18646288209606984), np.float64(0.2074235807860262), np.float64(0.1838427947598253), np.float64(0.1873362445414847), np.float64(0.18340611353711792), np.float64(0.185589519650655), np.float64(0.18296943231441049), np.float64(0.21528384279475982), np.float64(0.18471615720524018), np.float64(0.2109170305676856), np.float64(0.21877729257641917), np.float64(0.18384279475982535), np.float64(0.1873362445414847), np.float64(0.17903930131004367), np.float64(0.2165938864628821), np.float64(0.18165938864628822), np.float64(0.18602620087336244), np.float64(0.1834061135371179), np.float64(0.18165938864628822), np.float64(0.18165938864628822), np.float64(0.1851528384279476), np.float64(0.1895196506550218), np.float64(0.20524017467248906), np.float64(0.2091703056768559), np.float64(0.18646288209606984), np.float64(0.1868995633187773), np.float64(0.18602620087336244), np.float64(0.21397379912663755), np.float64(0.18995633187772926), np.float64(0.2126637554585153), np.float64(0.1890829694323144), np.float64(0.2183406113537118), np.float64(0.19344978165938861), np.float64(0.1851528384279476), np.float64(0.18471615720524018), np.float64(0.18296943231441049), np.float64(0.22445414847161574), np.float64(0.2165938864628821), np.float64(0.18384279475982537), np.float64(0.18209606986899562), np.float64(0.1903930131004367), np.float64(0.21703056768558954), np.float64(0.19126637554585152), np.float64(0.18820960698689956), np.float64(0.2165938864628821), np.float64(0.18253275109170308), np.float64(0.1838427947598253), np.float64(0.18427947598253275), np.float64(0.1925764192139738), np.float64(0.20524017467248906), np.float64(0.18209606986899565), np.float64(0.18078602620087336), np.float64(0.18384279475982532), np.float64(0.21135371179039303), np.float64(0.2240174672489083), np.float64(0.21310043668122267), np.float64(0.18340611353711792), np.float64(0.21484716157205236), np.float64(0.1895196506550218), np.float64(0.20480349344978163), np.float64(0.2109170305676856), np.float64(0.21528384279475982), np.float64(0.21659388646288208), np.float64(0.214410480349345), np.float64(0.21179039301310043), np.float64(0.17860262008733624), np.float64(0.18427947598253275), np.float64(0.20829694323144105), np.float64(0.20873362445414845), np.float64(0.24192139737991267), np.float64(0.2126637554585153)], 75: [np.float64(0.27853658536585374), np.float64(0.22731707317073174), np.float64(0.22439024390243906), np.float64(0.22487804878048784), np.float64(0.22780487804878052), np.float64(0.22682926829268296), np.float64(0.2234146341463415), np.float64(0.2282926829268293), np.float64(0.22634146341463418), np.float64(0.22439024390243906), np.float64(0.28390243902439033), np.float64(0.2204878048780488), np.float64(0.2209756097560976), np.float64(0.282439024390244), np.float64(0.22634146341463418), np.float64(0.2775609756097561), np.float64(0.288780487804878), np.float64(0.23024390243902443), np.float64(0.22390243902439028), np.float64(0.2282926829268293), np.float64(0.23106796116504852), np.float64(0.2234146341463415), np.float64(0.22487804878048784), np.float64(0.2282926829268293), np.float64(0.22390243902439028), np.float64(0.2209756097560976), np.float64(0.229126213592233), np.float64(0.2307317073170732), np.float64(0.22390243902439028), np.float64(0.2736585365853658), np.float64(0.2258536585365854), np.float64(0.22731707317073174), np.float64(0.2331707317073171), np.float64(0.27268292682926826), np.float64(0.22780487804878052), np.float64(0.23024390243902443), np.float64(0.22975609756097565), np.float64(0.22780487804878052), np.float64(0.22682926829268296), np.float64(0.21902439024390247), np.float64(0.2258536585365854), np.float64(0.2307317073170732), np.float64(0.27707317073170734), np.float64(0.22926829268292687), np.float64(0.22878048780487809), np.float64(0.23219512195121955), np.float64(0.2785365853658537), np.float64(0.22634146341463418), np.float64(0.21658536585365856), np.float64(0.2282926829268293), np.float64(0.2402912621359224), np.float64(0.22475728155339808), np.float64(0.23268292682926833), np.float64(0.2180487804878049), np.float64(0.23024390243902443), np.float64(0.2509708737864077), np.float64(0.28585365853658534), np.float64(0.23024390243902443), np.float64(0.23268292682926833), np.float64(0.2650485436893204), np.float64(0.2775609756097562), np.float64(0.23219512195121955), np.float64(0.2258536585365854), np.float64(0.2765853658536586), np.float64(0.22487804878048784), np.float64(0.22878048780487809), np.float64(0.22682926829268296), np.float64(0.2307317073170732), np.float64(0.22048780487804875), np.float64(0.22487804878048784), np.float64(0.2258536585365854), np.float64(0.22682926829268296), np.float64(0.2790243902439024), np.float64(0.271219512195122), np.float64(0.22341463414634144), np.float64(0.2209756097560976), np.float64(0.27804878048780496), np.float64(0.23024390243902443), np.float64(0.2936585365853658), np.float64(0.2834146341463415), np.float64(0.28048780487804875), np.float64(0.2463414634146342), np.float64(0.24731707317073176), np.float64(0.21756097560975612), np.float64(0.231219512195122), np.float64(0.22731707317073174), np.float64(0.2746341463414634), np.float64(0.22243902439024393), np.float64(0.2354368932038835), np.float64(0.2873170731707317)], 100: [np.float64(0.30994475138121547), np.float64(0.26077348066298345), np.float64(0.25745856353591157), np.float64(0.25082872928176797), np.float64(0.2458563535911602), np.float64(0.24861878453038674), np.float64(0.2486187845303868), np.float64(0.25745856353591157), np.float64(0.25635359116022094), np.float64(0.25193370165745854), np.float64(0.3110497237569061), np.float64(0.2519337016574586), np.float64(0.24640883977900555), np.float64(0.3044198895027624), np.float64(0.25524861878453037), np.float64(0.3149171270718231), np.float64(0.3082872928176796), np.float64(0.25082872928176797), np.float64(0.256353591160221), np.float64(0.25248618784530386), np.float64(0.28508287292817686), np.float64(0.2585635359116022), np.float64(0.26022099447513813), np.float64(0.2574585635359116), np.float64(0.2602209944751381), np.float64(0.25027624309392266), np.float64(0.2616666666666666), np.float64(0.24972375690607734), np.float64(0.2834254143646409), np.float64(0.3165745856353591), np.float64(0.24972375690607734), np.float64(0.24917127071823203), np.float64(0.2519337016574585), np.float64(0.31988950276243094), np.float64(0.2569060773480663), np.float64(0.2591160220994475), np.float64(0.2569060773480663), np.float64(0.25138121546961323), np.float64(0.2535911602209945), np.float64(0.2580110497237569), np.float64(0.2585635359116022), np.float64(0.2785714285714286), np.float64(0.30497237569060776), np.float64(0.25303867403314917), np.float64(0.25248618784530386), np.float64(0.2458563535911602), np.float64(0.31767955801104975), np.float64(0.2508287292817679), np.float64(0.2458563535911602), np.float64(0.25524861878453037), np.float64(0.26055555555555554), np.float64(0.2796703296703297), np.float64(0.25303867403314917), np.float64(0.25635359116022094), np.float64(0.24751381215469617), np.float64(0.2767955801104972), np.float64(0.31215469613259667), np.float64(0.25745856353591157), np.float64(0.24696132596685083), np.float64(0.28021978021978017), np.float64(0.3165745856353591), np.float64(0.2596685082872928), np.float64(0.2618784530386741), np.float64(0.3121546961325967), np.float64(0.25082872928176797), np.float64(0.2541436464088398), np.float64(0.25138121546961323), np.float64(0.2513812154696133), np.float64(0.28508287292817686), np.float64(0.2502762430939226), np.float64(0.25248618784530386), np.float64(0.25303867403314917), np.float64(0.3060773480662983), np.float64(0.32044198895027626), np.float64(0.28287292817679555), np.float64(0.2478021978021978), np.float64(0.3099447513812154), np.float64(0.243646408839779), np.float64(0.3060773480662983), np.float64(0.30883977900552484), np.float64(0.31215469613259667), np.float64(0.3066298342541437), np.float64(0.29226519337016577), np.float64(0.28232044198895023), np.float64(0.24751381215469612), np.float64(0.2596685082872928), np.float64(0.3082872928176795), np.float64(0.2850828729281768), np.float64(0.2883977900552487), np.float64(0.3077348066298342)], 150: [np.float64(0.37841726618705035), np.float64(0.36376811594202896), np.float64(0.3601449275362319), np.float64(0.35942028985507257), np.float64(0.35869565217391297), np.float64(0.3594202898550724), np.float64(0.3565217391304348), np.float64(0.36086956521739133), np.float64(0.35), np.float64(0.3644927536231884), np.float64(0.3820143884892086), np.float64(0.3572463768115942), np.float64(0.3579710144927537), np.float64(0.3820143884892086), np.float64(0.3673913043478261), np.float64(0.3841726618705036), np.float64(0.37194244604316545), np.float64(0.36666666666666664), np.float64(0.35797101449275354), np.float64(0.3615942028985507), np.float64(0.3496402877697841), np.float64(0.36086956521739133), np.float64(0.36304347826086947), np.float64(0.36521739130434777), np.float64(0.35869565217391297), np.float64(0.35942028985507235), np.float64(0.33695652173913043), np.float64(0.36159420289855077), np.float64(0.3510791366906475), np.float64(0.38129496402877694), np.float64(0.35942028985507246), np.float64(0.35869565217391297), np.float64(0.36304347826086947), np.float64(0.3820143884892086), np.float64(0.35869565217391297), np.float64(0.3572463768115942), np.float64(0.37028985507246376), np.float64(0.35), np.float64(0.36594202898550726), np.float64(0.3630434782608696), np.float64(0.363768115942029), np.float64(0.327536231884058), np.float64(0.37913669064748196), np.float64(0.3565217391304348), np.float64(0.36014492753623184), np.float64(0.3594202898550724), np.float64(0.38273381294964026), np.float64(0.3630434782608695), np.float64(0.3992753623188406), np.float64(0.35869565217391314), np.float64(0.33623188405797105), np.float64(0.39280575539568346), np.float64(0.355072463768116), np.float64(0.3630434782608695), np.float64(0.358695652173913), np.float64(0.3884892086330935), np.float64(0.38201438848920855), np.float64(0.3630434782608695), np.float64(0.3623188405797101), np.float64(0.3841726618705036), np.float64(0.3834532374100719), np.float64(0.3630434782608695), np.float64(0.3652173913043478), np.float64(0.3762589928057553), np.float64(0.36014492753623184), np.float64(0.3652173913043478), np.float64(0.3739130434782608), np.float64(0.36086956521739133), np.float64(0.339568345323741), np.float64(0.36231884057971014), np.float64(0.35869565217391297), np.float64(0.3695652173913043), np.float64(0.3712230215827338), np.float64(0.38345323741007187), np.float64(0.3503597122302159), np.float64(0.3536231884057971), np.float64(0.38848920863309355), np.float64(0.3536231884057971), np.float64(0.3848920863309352), np.float64(0.37841726618705035), np.float64(0.37913669064748207), np.float64(0.29782608695652174), np.float64(0.29565217391304344), np.float64(0.34532374100719426), np.float64(0.3615942028985507), np.float64(0.3644927536231884), np.float64(0.38129496402877694), np.float64(0.35467625899280575), np.float64(0.34785714285714286), np.float64(0.3776978417266187)], 200: [np.float64(0.4946236559139786), np.float64(0.48723404255319147), np.float64(0.48936170212765956), np.float64(0.4957446808510638), np.float64(0.4861702127659574), np.float64(0.4819148936170213), np.float64(0.4893617021276596), np.float64(0.4968085106382979), np.float64(0.48510638297872344), np.float64(0.4904255319148937), np.float64(0.4881720430107528), np.float64(0.4925531914893617), np.float64(0.48829787234042554), np.float64(0.4838709677419354), np.float64(0.47659574468085103), np.float64(0.4763440860215054), np.float64(0.49247311827956997), np.float64(0.4925531914893617), np.float64(0.48191489361702133), np.float64(0.4914893617021277), np.float64(0.46382978723404256), np.float64(0.4914893617021277), np.float64(0.473404255319149), np.float64(0.47659574468085103), np.float64(0.48191489361702133), np.float64(0.48510638297872344), np.float64(0.535483870967742), np.float64(0.4925531914893616), np.float64(0.5096774193548386), np.float64(0.4720430107526882), np.float64(0.498936170212766), np.float64(0.48510638297872344), np.float64(0.48510638297872344), np.float64(0.4774193548387096), np.float64(0.4840425531914893), np.float64(0.4840425531914894), np.float64(0.4925531914893616), np.float64(0.49787234042553197), np.float64(0.48936170212765956), np.float64(0.48085106382978715), np.float64(0.4914893617021276), np.float64(0.5989247311827958), np.float64(0.4838709677419354), np.float64(0.48191489361702133), np.float64(0.48191489361702133), np.float64(0.4861702127659574), np.float64(0.48064516129032253), np.float64(0.5053191489361701), np.float64(0.5170212765957446), np.float64(0.5063829787234042), np.float64(0.5440860215053764), np.float64(0.4127659574468085), np.float64(0.474468085106383), np.float64(0.4840425531914893), np.float64(0.48723404255319147), np.float64(0.546236559139785), np.float64(0.4903225806451612), np.float64(0.47765957446808505), np.float64(0.49255319148936183), np.float64(0.5276595744680852), np.float64(0.478494623655914), np.float64(0.4957446808510639), np.float64(0.48617021276595745), np.float64(0.4709677419354838), np.float64(0.48617021276595745), np.float64(0.48085106382978726), np.float64(0.4829787234042554), np.float64(0.48617021276595745), np.float64(0.5032258064516129), np.float64(0.4925531914893617), np.float64(0.49361702127659574), np.float64(0.48936170212765956), np.float64(0.49247311827956997), np.float64(0.4741935483870968), np.float64(0.5021505376344086), np.float64(0.5074468085106382), np.float64(0.4870967741935483), np.float64(0.48617021276595745), np.float64(0.4956989247311828), np.float64(0.5), np.float64(0.5032258064516129), np.float64(0.4602150537634408), np.float64(0.4473118279569892), np.float64(0.510752688172043), np.float64(0.49361702127659574), np.float64(0.473404255319149), np.float64(0.4946236559139785), np.float64(0.5064516129032258), np.float64(0.4806451612903226), np.float64(0.4903225806451614)]}
acc_with_singleton_avg_dict = {
    k: float(np.mean(v))
    for k, v in all_leiden_elegans_labels_with_singleton_acc_dict.items()
}
acc_with_singleton_avg_dict


{0: 0.16463958582238153,
 1: 0.16288569144684256,
 5: 0.16604622871046226,
 10: 0.17345311854605533,
 20: 0.16586872586872586,
 50: 0.19592916060164964,
 75: 0.23979530086563006,
 100: 0.27086776909060517,
 150: 0.36334445238839985,
 200: 0.49045768829914344}